In [10]:
import numpy as np
import pandas as pd
import json
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import sys
sys.path.append('..')

from helpers.save_model_params import extract_bagging_to_dict, extract_boosting_to_dict
from helpers.tfidf_vectorizer import TfidfVectorizer
from models.decision_tree import BaggingTreeModel, BoostingTreeModel
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [11]:
random_seed = 42

In [12]:
# create a simple numeric classification dataset with 3 classes
X, y = make_classification(
    n_samples=300,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    n_clusters_per_class=1,
    n_classes=3,
    random_state=42
)

# put into a DataFrame for convenience
feature_names = [f"f{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["label"] = y

# stratified train/test split
train_df, test_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)

print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
print("Class distribution in train:\n", train_df["label"].value_counts().sort_index())

X_train = train_df[feature_names].values
y_train = train_df["label"].values
X_test = test_df[feature_names].values
y_test = test_df["label"].values

Train shape: (210, 7) Test shape: (90, 7)
Class distribution in train:
 label
0    69
1    69
2    72
Name: count, dtype: int64


In [13]:
# Create base estimator
base_tree = DecisionTreeClassifier(
    max_depth=7,
    min_samples_split=2,
    random_state=random_seed,
    criterion='gini'
)

# Create BaggingClassifier
bagging_model = BaggingClassifier(
    estimator=base_tree,
    n_estimators=20,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    bootstrap_features=False,
    oob_score=True,
    random_state=random_seed,
    n_jobs=2,
    verbose=1
)

In [14]:
bagging_model.fit(X_train, y_train)

y_pred = bagging_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("BaggingClassifier (20 trees) accuracy:", acc)
print(classification_report(y_test, y_pred))

print("prediction stats:", np.unique(y_pred, return_counts=True))

BaggingClassifier (20 trees) accuracy: 0.8111111111111111
              precision    recall  f1-score   support

           0       0.86      0.83      0.85        30
           1       0.85      0.77      0.81        30
           2       0.74      0.83      0.78        30

    accuracy                           0.81        90
   macro avg       0.82      0.81      0.81        90
weighted avg       0.82      0.81      0.81        90

prediction stats: (array([0, 1, 2]), array([29, 27, 34]))


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.0s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.0s finished


In [15]:
# Extract model to dictionary
bagging_dict = extract_bagging_to_dict(bagging_model)

# Save to JSON file
bagging_output_path = '../models/fitted_bagging.json'
with open(bagging_output_path, 'w') as f:
    json.dump(bagging_dict, f, indent=2)

print(f"Bagging model saved to: {bagging_output_path}")
print(f"\nModel metadata:")
for key, value in bagging_dict['metadata'].items():
    if key != 'feature_names' and key != 'classes':
        print(f"  {key}: {value}")

Bagging model saved to: ../models/fitted_bagging.json

Model metadata:
  model_type: BaggingClassifier
  n_estimators: 20
  n_features: 6
  n_classes: 3
  max_samples: 0.8
  max_features: 0.8
  bootstrap: True
  bootstrap_features: False
  oob_score: True
  warm_start: False
  random_state: 42
  oob_score_value: 0.7904761904761904


In [16]:
custom_bagging = BaggingTreeModel('../models/fitted_bagging.json')

custom_bagging_train_pred = custom_bagging.predict(X_train)
custom_bagging_val_pred = custom_bagging.predict(X_test)

print(X_train)

print("prediction stats:", np.unique(custom_bagging_val_pred, return_counts=True))

print("Comparison on test set:")
print("Custom Bagging Model Accuracy:", accuracy_score(y_test, custom_bagging_val_pred))
print("Sklearn Bagging Model Accuracy:", accuracy_score(y_test, y_pred))

Bagging ensemble loaded from: ../models/fitted_bagging.json
Number of trees: 20
[[ 0.27670904 -0.63070965 -0.57349818  0.73214056  1.77246626  0.42247943]
 [ 0.2450186  -0.11702078 -1.18108108 -0.19742954  0.34058138  1.73183492]
 [ 0.12323027 -1.10484589 -0.4704943   0.70946182 -1.06199569 -1.51860717]
 ...
 [-0.5376897  -1.12274063 -1.09630052  0.30344475 -1.75732868  0.6976982 ]
 [ 1.03823959  0.21226365  0.94080863  0.08636757  2.56459581  1.52284147]
 [ 0.82932482 -1.447988   -1.53793767 -0.58183783  0.78743984  0.85433274]]
prediction stats: (array([0, 1, 2]), array([29, 27, 34]))
Comparison on test set:
Custom Bagging Model Accuracy: 0.8111111111111111
Sklearn Bagging Model Accuracy: 0.8111111111111111
